# Tool Calling from Scratch


In [76]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage, 
    SystemMessage, 
    ToolMessage
)
from langchain.tools import tool
from dotenv import load_dotenv

In [77]:
load_dotenv()

True

In [78]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

In [79]:
@tool
def mult(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

In [80]:
tools = [mult]
tool_map = {tool.name: tool for tool in tools}
tool_map

{'mult': StructuredTool(name='mult', description='Multiplies two numbers.', args_schema=<class 'langchain_core.utils.pydantic.mult'>, func=<function mult at 0x7cf985fe4b40>)}

In [81]:
llm_with_tools = llm.bind_tools(tools)

In [82]:
question = "What is 3 multiplied by 4?"

In [83]:
messages = [
    SystemMessage("You are a helpful assistant that can use tools to answer questions."),
    HumanMessage(question)
]

In [84]:
ai_message = llm_with_tools.invoke(messages)

In [85]:
ai_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 68, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4dafa39984', 'id': 'chatcmpl-DoDVcoRmkezY93yUfclJEVCof87VL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ea391-c248-7141-b1a9-583ec3cc4888-0', tool_calls=[{'name': 'mult', 'args': {'a': 3, 'b': 4}, 'id': 'call_UH6VcRxd7I2Jl4QuqQWB0CvY', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 68, 'output_tokens': 17, 'total_tokens': 85, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [86]:
messages.append(ai_message)
messages

[SystemMessage(content='You are a helpful assistant that can use tools to answer questions.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 3 multiplied by 4?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 68, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4dafa39984', 'id': 'chatcmpl-DoDVcoRmkezY93yUfclJEVCof87VL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ea391-c248-7141-b1a9-583ec3cc4888-0', tool_calls=[{'name': 'mult', 'args': {'a': 3, 'b': 4}, 'id': 'call_UH6VcRxd7I2Jl4QuqQWB0CvY', 'type': 'tool_call

In [87]:
for tool_call in ai_message.tool_calls:
    tool_call_id = tool_call['id']
    function_name = tool_call['name']
    arguments = tool_call['args']
    func = tool_map[function_name]
    result = func.invoke(arguments)
    tool_message = ToolMessage(
        content=result,
        name=function_name,
        tool_call_id=tool_call_id,
    )
    messages.append(tool_message)

In [88]:
messages

[SystemMessage(content='You are a helpful assistant that can use tools to answer questions.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 3 multiplied by 4?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 68, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4dafa39984', 'id': 'chatcmpl-DoDVcoRmkezY93yUfclJEVCof87VL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ea391-c248-7141-b1a9-583ec3cc4888-0', tool_calls=[{'name': 'mult', 'args': {'a': 3, 'b': 4}, 'id': 'call_UH6VcRxd7I2Jl4QuqQWB0CvY', 'type': 'tool_call

In [89]:
ai_message = llm_with_tools.invoke(messages)

In [90]:
ai_message

AIMessage(content='3 multiplied by 4 is 12.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 93, 'total_tokens': 103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4dafa39984', 'id': 'chatcmpl-DoDVdUHQ0psDjncX5QVDGRxrzpL7U', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ea391-c942-78b3-846f-3af0839c3d87-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 10, 'total_tokens': 103, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})